# Feature Engineering
Prepare the dataset for training by cleaning, encoding, and saving the processed version.

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

def prepare_features(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df.copy()
    df.columns = df.columns.str.strip()
    df = df.drop(columns=[
        "Year",
        "Employment_Rate_6_Months (%)",
        "Average_Starting_Salary_USD",
        "Top_Industry",
        "Job_Role",
        "University_Name",
        "Country",
        "Region"
    ], errors="ignore")

    target = df["Employment_Rate_12_Months (%)"].copy()
    risk_cutoff = target.quantile(0.25)
    df["Unemployability_Risk"] = (target <= risk_cutoff).astype(int)

    degree_map = {"Bachelor": 1, "Master": 2, "PhD": 3}
    df["Degree_Level_Ordinal"] = df["Degree_Level"].map(degree_map).fillna(0).astype(int)
    df["Graduation_Recency"] = df["Graduation_Year"] - df["Graduation_Year"].min()
    df["Demand_x_Reputation"] = df["Skill_Demand_Score (1–100)"] * df["Employer_Reputation_Score (1–100)"]
    df["Demand_x_Remote"] = df["Skill_Demand_Score (1–100)"] * df["Remote_Work_Availability (%)"]
    df["Reputation_x_Remote"] = df["Employer_Reputation_Score (1–100)"] * df["Remote_Work_Availability (%)"]

    skill_encoder = LabelEncoder()
    for skill_col in ["Skill_1", "Skill_2", "Skill_3"]:
        df[skill_col] = skill_encoder.fit_transform(df[skill_col].astype(str))

    df = pd.get_dummies(df, columns=["Field_of_Study"], drop_first=True, dtype=int)

    scaler = StandardScaler()
    df["Skill_Demand_Score (1–100)"] = scaler.fit_transform(df[["Skill_Demand_Score (1–100)"]])

    return df

raw_data = pd.read_csv("../data/raw/global_graduate_employability_index.csv")
processed_data = prepare_features(raw_data)
processed_data.to_csv("../data/processed/cleaned_data.csv", index=False)
processed_data.head()


,Degree_Level,Graduation_Year,Employment_Rate_12_Months (%),Skill_1,Skill_2,Skill_3,Skill_Demand_Score (1–100),Remote_Work_Availability (%),Employer_Reputation_Score (1–100),Unemployability_Risk,...,Graduation_Recency,Demand_x_Reputation,Demand_x_Remote,Reputation_x_Remote,Field_of_Study_Computer Science,Field_of_Study_Data Science & AI,Field_of_Study_Engineering,Field_of_Study_Healthcare & Medicine,Field_of_Study_Natural Sciences,Field_of_Study_Social Sciences
0,Bachelor,2017,85.6,0,16,17,0.259208,8.8,66,0,...,2,4554,607.2,580.8,0,0,1,0,0,0
1,Bachelor,2023,87.9,16,0,17,0.395184,65.4,63,0,...,8,4473,4643.4,4120.2,0,0,1,0,0,0
2,Master,2019,83.2,2,10,22,-0.896584,5.0,74,1,...,4,3848,260.0,370.0,0,0,0,1,0,0
3,Master,2016,92.1,9,25,3,0.259208,10.3,48,0,...,1,3312,710.7,494.4,1,0,0,0,0,0
4,PhD,2023,86.3,13,6,19,0.259208,64.0,65,0,...,8,4485,4416.0,4160.0,0,0,0,0,0,0


In [2]:
print(f"Processed rows: {processed_data.shape[0]}")
print(f"Processed features: {processed_data.shape[1]}")
print("Saved processed data to data/processed/cleaned_data.csv")


Processed rows: 3500
Processed features: 21
Saved processed data to data/processed/cleaned_data.csv
